# 11 · Triangular faults

Every fault so far has been a flat rectangle, or a grid of them. Real faults
bend, branch, and follow curved slab surfaces that rectangles tile awkwardly,
leaving gaps or overlaps at the corners. Triangles tile any surface cleanly.
This chapter swaps the rectangular source for a triangular one and shows that
almost nothing else about the inverse problem changes: it is still
$\mathbf{d} = G\mathbf{m}$, solved and assessed exactly as before.

**Learning objectives**

By the end of the chapter, you will be able to:

- explain when triangular elements are preferable to rectangles
- build a small curved triangular fault from nodes and connectivity
- reuse the same Green's assembly, inversion, and plotting on a mesh
- choose a geographic azimuth basis when patch orientation varies

**Prerequisites:** Chapters 02 and 07
**Estimated time:** 30–60 minutes

GeoDef's axes, signs, and units are summarized in
[the conventions guide](../docs/conventions.md). New terms are introduced in
words here and collected in [the glossary](../docs/glossary.md).

## 1. Why triangles

A rectangle is the natural tile for a flat plane, but faults are rarely flat.
A subduction megathrust curves along the top of a sinking slab; a strike-slip
fault steps and bends along its trace. Forcing rectangles onto such a surface
either leaves gaps between tiles or makes them overlap, and both corrupt the
modeled displacement.

Triangles solve this. Any surface, however curved, can be covered by triangles
with no gaps and no overlaps, because three points always define a flat patch.
The price is bookkeeping: unlike a tidy grid, a triangular mesh has no simple
row-and-column order, and each triangle can point in its own direction. The
benefit is that everything downstream of building $G$, the inversion, the
regularization, the uncertainty and resolution analysis, works unchanged.

## 2. The same operator, a different source

GeoDef builds the columns of $G$ for a triangle using the Nikkhoo and Walter
(2015) triangular dislocation, an artifact-free elastic solution analogous to
Okada's rectangle. Each triangle carries constant slip, just as each rectangle
did, so a column of $G$ is still one patch's surface footprint (Chapter 02).
Only the formula that fills those columns changes:

$$ \mathbf{d} = G_{\text{tri}}\,\mathbf{m} + \boldsymbol{\varepsilon}. $$

The projection into a dataset, the covariance weighting, the regularization,
and the posterior covariance are all identical to the rectangular case. One
practical difference: on a grid the smoothing Laplacian is a simple stencil,
but on an unstructured mesh GeoDef builds it from each triangle's actual
neighbors.

## 3. Build a small curved mesh

A triangular fault is defined by two things: a list of **nodes** (corner points
in space) and a list of **triangles** (which three nodes form each patch). We
build a small nine-node mesh tied to a local frame at 34 N, 118 W.

The nodes are given in local East, North, Up meters. Note the sign of the third
column: `up` is *negative* at depth, so these nodes sit below the surface and
deepen toward the north.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import geodef

%matplotlib inline

rng = np.random.default_rng(0)
frame = geodef.LocalFrame(34.0, -118.0)

In [ ]:
# Nine nodes on a 3 x 3 patch of fault, in local East, North, Up meters.
# Up is negative at depth; the surface deepens (more negative) toward the north.
nodes = np.array([
    [-12_000.0, -6_000.0,  -3_000.0],
    [      0.0, -6_000.0,  -4_000.0],
    [ 12_000.0, -6_000.0,  -5_500.0],
    [-12_000.0,      0.0,  -5_500.0],
    [      0.0,      0.0,  -6_800.0],
    [ 12_000.0,      0.0,  -8_500.0],
    [-12_000.0,  6_000.0,  -8_000.0],
    [      0.0,  6_000.0,  -9_500.0],
    [ 12_000.0,  6_000.0, -12_000.0],
])

In [ ]:
# Each row lists the three node indices forming one triangle: eight in all.
triangles = np.array([
    [0, 1, 3], [1, 4, 3], [1, 2, 4], [2, 5, 4],
    [3, 4, 6], [4, 7, 6], [4, 5, 7], [5, 8, 7],
])
fault = geodef.Fault.from_triangles(nodes, frame=frame, triangles=triangles)
print(fault.validate())

In [ ]:
geodef.plot.fault3d(fault, color_by="depth", aspect=2.0)
plt.show()

## 4. Put slip on the mesh, with a geographic direction

The mesh bends: because each triangle has its own strike and dip, a single
*local* rake (Chapter 07) would point in different geographic directions on
different patches. When we want one consistent geographic slip direction, an
**azimuth** basis is clearer. We prescribe true slip pointing due east (azimuth
90) and let `geodef.slip.from_azimuth` translate that into the local strike and
dip components each patch needs.

In [ ]:
# One signed amplitude per triangle, all slipping toward geographic east.
amplitude_true = np.array([0.15, 0.35, 0.65, 0.95, 0.3, 0.7, 0.9, 0.4])
strike_true, dip_true = geodef.slip.from_azimuth(
    amplitude_true,
    azimuth_degrees=90.0,
    fault_strike_degrees=fault.strike,   # per-patch strike from the mesh
)

Forward-model that slip to a small GNSS network and add 3 mm noise, the same
synthetic-data recipe as earlier chapters.

In [ ]:
station_lon, station_lat = np.meshgrid(
    np.linspace(-118.18, -117.82, 6), np.linspace(33.90, 34.16, 5)
)
station_lon, station_lat = station_lon.ravel(), station_lat.ravel()
east, north, up = fault.displacement(
    station_lat, station_lon, slip_strike=strike_true, slip_dip=dip_true
)
sigma = 0.003
gnss = geodef.data.gnss(
    lon=station_lon, lat=station_lat,
    east=east + rng.normal(0.0, sigma, east.size),
    north=north + rng.normal(0.0, sigma, north.size),
    up=up + rng.normal(0.0, sigma, up.size),
    sigma_east=sigma, sigma_north=sigma, sigma_up=sigma,
    name="triangular_gnss",
)

## 5. Invert with the familiar call

Here is the payoff. The solve is the *same* `geodef.solve` used for rectangles,
with the same regularization and non-negativity options. We fix the slip
azimuth to 90 degrees to match the truth and bound the amplitude to be
non-negative.

In [ ]:
result = geodef.solve(
    fault, datasets=gnss, components="azimuth", slip_azimuth=90.0,
    regularization="laplacian", regularization_strength=0.1,
    bounds=(0.0, None),
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4), constrained_layout=True)
geodef.plot.slip(fault, amplitude_true, ax=axes[0],
                 title="Input amplitude", colorbar_label="Slip (m)")
geodef.plot.slip_interpolated(fault, result.slip_magnitude, ax=axes[1],
                              title="Recovered (interpolated)",
                              colorbar_label="Slip (m)")
plt.show()

The recovered pattern matches the input. One caution about the right panel:
`slip_interpolated` draws a smooth surface for readability, but the actual
estimate is still just eight constant-slip triangles. The smoothness is a
drawing choice, not extra resolution; the honest result is the eight patch
values.

**Checkpoint.** We passed `fault.strike` (an array, one value per triangle) to
`from_azimuth`, but a single `azimuth_degrees=90`. Why does the azimuth stay a
scalar while the strike is per-patch?

<details><summary>Answer</summary>

The azimuth is the *geographic* direction we want slip to point, and we want
the same direction everywhere, so it is one number. Each triangle has a
different orientation, so translating that one geographic direction into
local strike and dip components requires each patch's own strike. The function
does that translation patch by patch.

</details>

## Checkpoint questions

**Which parts of the inverse problem changed when we switched to triangles?**

<details><summary>Answer</summary>

Only the source geometry and the dislocation kernel that fills the columns of
$G$. The linear algebra, the covariance weighting, the constraints, and the
uncertainty and resolution analysis are all unchanged.

</details>

**Why prefer an azimuth basis over a single rake on this mesh?**

<details><summary>Answer</summary>

Because the triangles have different strikes. One local rake would therefore
point in different geographic directions across the mesh, whereas one geographic
azimuth stays consistent and is translated into the right local components on
each patch.

</details>

**Does a smooth interpolated slip plot mean the slip is smoothly resolved?**

<details><summary>Answer</summary>

No. It is a rendering of a handful of discrete patch values. The interpolation
adds visual smoothness, not information; always remember the estimate is the
per-patch numbers.

</details>

## Common mistakes

- **Getting the depth sign wrong.** Local `up` is negative below the surface.
  Supplying positive-down depths as the `z` node coordinate places triangles
  *above* the ground.
- **Mixing local frames.** Nodes built in one `LocalFrame` and combined with
  another silently corrupt the geometry. Keep a mesh in a single frame.
- **Smoothing across a sharp bend.** Applying a Laplacian to triangle-local
  strike and dip components across a sudden change in orientation imposes an
  artificial direction. Use a geographic basis, or break the smoothing at the
  bend.

## Recap

- Triangles tile curved and irregular fault surfaces cleanly, where rectangles
  leave gaps or overlaps.
- Only the dislocation kernel changes; the forward, solve, and assessment
  workflow is geometry-independent.
- A geographic azimuth basis keeps the slip direction consistent when patch
  orientation varies.
- Interpolated slip plots are for readability; the estimate is the per-patch
  values.

Chapter 12 applies the same machinery to the interseismic problem, imaging how
strongly a fault is locked between earthquakes.

## Exercises

Predict the qualitative outcome before running each experiment. Worked
solutions are in `solutions/11_triangular_faults_solutions.ipynb`.

1. **Refinement.** Split each triangle into smaller ones and compare how the
   prediction and the conditioning change.
2. **Direction bases.** Solve with a fixed rake and with a fixed azimuth, and
   map the geographic slip direction each implies across the bending mesh.
3. **Spike tests.** Put a unit spike on each triangle in turn, invert, and
   compare how well localization holds as the patches deepen.
4. **Challenge: a bent join.** Add a second segment meeting the first at a
   sharp bend, and discuss whether the smoothing Laplacian should connect
   patches across the join.

## Further reading

- Nikkhoo, M., and Walter, T. R. (2015), "Triangular dislocation: an
  analytical, artefact-free solution," *Geophysical Journal International*,
  201(2), 1119–1141.
- Meade, B. J. (2007), "Algorithms for the calculation of exact displacements,
  strains, and stresses for triangular dislocation elements," *Computers &
  Geosciences*, 33(8), 1064–1075.
- `examples/mesh_generation.ipynb` for practical trace, polygon, and slab2.0
  meshing.
- [GeoDef mesh documentation](../docs/mesh.md) and
  [fault documentation](../docs/fault.md).
- The next course chapter is `12_interseismic_coupling.ipynb`.